# HuggingFace Model Optimization Techniques
*Early Production Stage: Preparing Models for Deployment*

This notebook covers essential optimization techniques to make HuggingFace models production-ready, focusing on performance, memory efficiency, and deployment considerations.

## Learning Objectives
- Understand different quantization methods
- Implement model compression techniques
- Optimize inference performance
- Prepare models for deployment
- Balance model size vs quality trade-offs

## Prerequisites
- Trained or pre-trained HuggingFace model
- GPU recommended for optimization
- Understanding of basic model concepts

## Setup and Environment

In [ ]:
# Install required packages (run in terminal if needed)
# pip install transformers torch accelerate bitsandbytes auto-gptq optimum

import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GPTQConfig
)
import bitsandbytes as bnb
from optimum.gptq import GPTQQuantizer
import psutil
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# System memory info
memory = psutil.virtual_memory()
print(f"System RAM: {memory.total / 1024**3:.1f} GB")
print(f"Available RAM: {memory.available / 1024**3:.1f} GB")

## Understanding Model Optimization

Why and when to optimize models for production deployment.

In [ ]:
OPTIMIZATION_CONCEPTS = {
    "quantization": {
        "description": "Reduce model precision from FP32 to lower precision",
        "benefits": ["Smaller model size", "Faster inference", "Lower memory usage"],
        "trade_offs": ["Potential quality loss", "May require calibration"]
    },
    "pruning": {
        "description": "Remove unnecessary model weights",
        "benefits": ["Reduced model size", "Faster inference"],
        "trade_offs": ["Can affect accuracy", "Requires fine-tuning after pruning"]
    },
    "knowledge_distillation": {
        "description": "Train smaller model to mimic larger model behavior",
        "benefits": ["Maintain performance with smaller model", "Faster inference"],
        "trade_offs": ["Requires training data", "Time-consuming process"]
    },
    "architecture_optimization": {
        "description": "Modify model architecture for efficiency",
        "benefits": ["Better hardware utilization", "Optimized for specific tasks"],
        "trade_offs": ["May lose some generalization", "Architecture changes needed"]
    }
}

print("🎯 Model Optimization Concepts:")
for concept, details in OPTIMIZATION_CONCEPTS.items():
    print(f"\n{concept.upper().replace('_', ' ')}:")
    print(f"  Description: {details['description']}")
    print(f"  Benefits: {', '.join(details['benefits'])}")
    print(f"  Trade-offs: {', '.join(details['trade_offs'])}")

## Quantization Techniques

Learn different quantization approaches and their implementation.

In [ ]:
def load_base_model(model_name="gpt2"):
    """
    Load a base model for optimization experiments.
    """
    print(f"📥 Loading base model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # Calculate model size
    param_count = sum(p.numel() for p in model.parameters())
    model_size_mb = param_count * 4 / (1024**2)  # FP32 = 4 bytes per param
    
    print(f"Parameters: {param_count:,}")
    print(f"Model size (FP32): {model_size_mb:.1f} MB")
    
    return tokenizer, model, model_size_mb

# Load base model
tokenizer, base_model, base_size = load_base_model("gpt2")

def benchmark_model(model, tokenizer, prompt="Hello, how are you?", num_runs=3):
    """
    Benchmark model inference performance.
    """
    print(f"\n⚡ Benchmarking model...")
    
    # Prepare input
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model = model.cuda()
    
    # Warm up
    with torch.no_grad():
        _ = model.generate(**inputs, max_length=50, num_return_sequences=1, do_sample=False)
    
    # Benchmark
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start_time = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
    end_time = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
    
    if torch.cuda.is_available():
        start_time.record()
    
    import time
    cpu_start = time.time()
    
    for _ in range(num_runs):
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=50, num_return_sequences=1, do_sample=False)
    
    if torch.cuda.is_available():
        end_time.record()
        torch.cuda.synchronize()
        gpu_time = start_time.elapsed_time(end_time) / 1000  # Convert to seconds
        avg_time = gpu_time / num_runs
        device = "GPU"
    else:
        cpu_time = time.time() - cpu_start
        avg_time = cpu_time / num_runs
        device = "CPU"
    
    # Get memory usage
    if torch.cuda.is_available():
        memory_used = torch.cuda.memory_allocated() / (1024**3)  # GB
    else:
        memory_used = psutil.Process().memory_info().rss / (1024**3)  # GB
    
    # Generate sample output
    with torch.no_grad():
        sample_output = model.generate(**inputs, max_length=50, num_return_sequences=1, do_sample=False)
    sample_text = tokenizer.decode(sample_output[0], skip_special_tokens=True)
    
    results = {
        "avg_inference_time": avg_time,
        "memory_used_gb": memory_used,
        "device": device,
        "sample_output": sample_text
    }
    
    print(f"  Average inference time: {avg_time:.3f} seconds")
    print(f"  Memory used: {memory_used:.2f} GB ({device})")
    print(f"  Sample output: {sample_text[:100]}...")
    
    return results

# Benchmark base model
print("📊 Base Model Benchmark:")
base_benchmark = benchmark_model(base_model, tokenizer)

## 4-bit Quantization with BitsAndBytes

In [ ]:
def quantize_with_bitsandbytes(model_name="gpt2"):
    """
    Apply 4-bit quantization using BitsAndBytes.
    """
    print(f"\n🔢 Quantizing {model_name} with BitsAndBytes (4-bit)...")
    
    # Configure quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"  # Normalized Float 4
    )
    
    # Load quantized model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    quantized_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"  # Automatically distribute across available devices
    )
    
    # Calculate quantized model size
    param_count = sum(p.numel() for p in quantized_model.parameters())
    # 4-bit quantization = 0.5 bytes per parameter
    quantized_size_mb = param_count * 0.5 / (1024**2)
    
    print(f"Quantized parameters: {param_count:,}")
    print(f"Quantized model size: {quantized_size_mb:.1f} MB")
    print(f"Size reduction: {base_size / quantized_size_mb:.1f}x")
    
    return tokenizer, quantized_model, quantized_size_mb

# Apply 4-bit quantization
bnb_tokenizer, bnb_model, bnb_size = quantize_with_bitsandbytes("gpt2")

# Benchmark quantized model
print("\n📊 4-bit Quantized Model Benchmark:")
bnb_benchmark = benchmark_model(bnb_model, bnb_tokenizer)

## 8-bit Quantization

In [ ]:
def quantize_with_8bit(model_name="gpt2"):
    """
    Apply 8-bit quantization.
    """
    print(f"\n🔢 Quantizing {model_name} with 8-bit quantization...")
    
    # Configure 8-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16
    )
    
    # Load quantized model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    quantized_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
    
    # Calculate size
    param_count = sum(p.numel() for p in quantized_model.parameters())
    quantized_size_mb = param_count * 1 / (1024**2)  # 8-bit = 1 byte per param
    
    print(f"8-bit quantized model size: {quantized_size_mb:.1f} MB")
    print(f"Size reduction: {base_size / quantized_size_mb:.1f}x")
    
    return tokenizer, quantized_model, quantized_size_mb

# Apply 8-bit quantization
bit8_tokenizer, bit8_model, bit8_size = quantize_with_8bit("gpt2")

# Benchmark 8-bit model
print("\n📊 8-bit Quantized Model Benchmark:")
bit8_benchmark = benchmark_model(bit8_model, bit8_tokenizer)

## GPTQ Quantization (Post-Training Quantization)

In [ ]:
def quantize_with_gptq(model_name="gpt2", bits=4):
    """
    Apply GPTQ quantization for better performance.
    Note: This requires significant computational resources.
    """
    print(f"\n🎯 GPTQ {bits}-bit quantization for {model_name}")
    print("Note: GPTQ quantization can be computationally intensive...")
    
    try:
        # Configure GPTQ
        gptq_config = GPTQConfig(
            bits=bits,
            dataset="c4",  # Calibration dataset
            tokenizer=model_name,
            use_fast=True,
            damp_percent=0.01,
            blocksize=128,
            pad_token_id=None
        )
        
        # Load and quantize model
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        gptq_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=gptq_config,
            device_map="auto"
        )
        
        # Calculate size
        param_count = sum(p.numel() for p in gptq_model.parameters())
        quantized_size_mb = param_count * (bits/8) / (1024**2)
        
        print(f"GPTQ {bits}-bit model size: {quantized_size_mb:.1f} MB")
        print(f"Size reduction: {base_size / quantized_size_mb:.1f}x")
        
        return tokenizer, gptq_model, quantized_size_mb
        
    except Exception as e:
        print(f"GPTQ quantization failed: {e}")
        print("This is common for smaller models or limited hardware.")
        return None, None, None

# Try GPTQ quantization (may fail on limited hardware)
gptq_tokenizer, gptq_model, gptq_size = quantize_with_gptq("gpt2", bits=4)

if gptq_model is not None:
    print("\n📊 GPTQ Quantized Model Benchmark:")
    gptq_benchmark = benchmark_model(gptq_model, gptq_tokenizer)
else:
    gptq_benchmark = None

## Performance Comparison

Compare all optimization approaches.

In [ ]:
def create_comparison_table():
    """
    Create a comprehensive comparison of all optimization methods.
    """
    results = []
    
    # Base model
    results.append({
        "method": "FP32 (Base)",
        "size_mb": base_size,
        "size_reduction": 1.0,
        "inference_time": base_benchmark["avg_inference_time"],
        "memory_gb": base_benchmark["memory_used_gb"],
        "device": base_benchmark["device"]
    })
    
    # 8-bit quantization
    results.append({
        "method": "8-bit Quantization",
        "size_mb": bit8_size,
        "size_reduction": base_size / bit8_size,
        "inference_time": bit8_benchmark["avg_inference_time"],
        "memory_gb": bit8_benchmark["memory_used_gb"],
        "device": bit8_benchmark["device"]
    })
    
    # 4-bit quantization
    results.append({
        "method": "4-bit Quantization",
        "size_mb": bnb_size,
        "size_reduction": base_size / bnb_size,
        "inference_time": bnb_benchmark["avg_inference_time"],
        "memory_gb": bnb_benchmark["memory_used_gb"],
        "device": bnb_benchmark["device"]
    })
    
    # GPTQ (if successful)
    if gptq_benchmark:
        results.append({
            "method": "GPTQ 4-bit",
            "size_mb": gptq_size,
            "size_reduction": base_size / gptq_size,
            "inference_time": gptq_benchmark["avg_inference_time"],
            "memory_gb": gptq_benchmark["memory_used_gb"],
            "device": gptq_benchmark["device"]
        })
    
    # Create DataFrame
    df = pd.DataFrame(results)
    
    # Display results
    print("📊 Model Optimization Comparison:")
    print("=" * 80)
    print(df.to_string(index=False, float_format='%.2f'))
    
    # Summary insights
    print("\n💡 Key Insights:")
    best_size = df.loc[df['size_mb'].idxmin()]
    best_speed = df.loc[df['inference_time'].idxmin()]
    
    print(f"  • Best size reduction: {best_size['method']} ({best_size['size_reduction']:.1f}x smaller)")
    print(f"  • Fastest inference: {best_speed['method']} ({best_speed['inference_time']:.3f}s)")
    
    if len(df) > 1:
        avg_reduction = df['size_reduction'].mean()
        print(f"  • Average size reduction: {avg_reduction:.1f}x")
    
    return df

# Create and display comparison
comparison_df = create_comparison_table()

## Model Export and Deployment Preparation

Prepare optimized models for deployment.

In [ ]:
def prepare_model_for_deployment(model, tokenizer, method_name, output_dir="./optimized_models"):
    """
    Prepare and save an optimized model for deployment.
    """
    output_path = Path(output_dir) / method_name
    output_path.mkdir(parents=True, exist_ok=True)
    
    print(f"💾 Saving {method_name} model to {output_path}")
    
    # Save model and tokenizer
    model.save_pretrained(output_path)
    tokenizer.save_pretrained(output_path)
    
    # Calculate final size
    import os
    total_size = 0
    for file_path in output_path.rglob("*"):
        if file_path.is_file():
            total_size += file_path.stat().st_size
    
    size_mb = total_size / (1024**2)
    print(f"Saved model size: {size_mb:.1f} MB")
    
    # Create deployment info
    info = {
        "method": method_name,
        "model_size_mb": size_mb,
        "original_model": tokenizer.name_or_path,
        "quantization": getattr(model.config, 'quantization_config', None),
        "torch_dtype": str(model.dtype),
        "device_map": getattr(model, 'hf_device_map', 'cpu')
    }
    
    # Save info
    import json
    with open(output_path / "deployment_info.json", "w") as f:
        json.dump(info, f, indent=2, default=str)
    
    print(f"Deployment info saved to {output_path / 'deployment_info.json'}")
    
    return output_path, size_mb

# Prepare models for deployment
print("🚀 Preparing models for deployment...")

models_to_save = [
    (base_model, tokenizer, "base_fp32"),
    (bit8_model, bit8_tokenizer, "8bit_quantized"),
    (bnb_model, bnb_tokenizer, "4bit_quantized")
]

if gptq_model:
    models_to_save.append((gptq_model, gptq_tokenizer, "gptq_4bit"))

saved_models = []
for model, tok, name in models_to_save:
    try:
        path, size = prepare_model_for_deployment(model, tok, name)
        saved_models.append({"name": name, "path": path, "size_mb": size})
    except Exception as e:
        print(f"Failed to save {name}: {e}")

print("\n📁 Saved Models Summary:")
for model_info in saved_models:
    print(f"  • {model_info['name']}: {model_info['size_mb']:.1f} MB")

## Optimization Best Practices

Key recommendations for model optimization.

In [ ]:
OPTIMIZATION_BEST_PRACTICES = {
    "quantization_selection": [
        "Use 4-bit quantization for maximum size reduction",
        "Use 8-bit for better quality with good compression",
        "Consider GPTQ for best performance/accuracy trade-off",
        "Test quantization impact on your specific use case"
    ],
    "hardware_considerations": [
        "Match quantization to your deployment hardware",
        "Consider GPU memory limits for large models",
        "Test inference latency requirements",
        "Balance model size with loading time"
    ],
    "quality_preservation": [
        "Always benchmark quantized models on your data",
        "Use calibration datasets for quantization-aware training",
        "Monitor for quality degradation",
        "Consider fine-tuning after quantization"
    ],
    "deployment_readiness": [
        "Save models with proper configuration",
        "Document quantization settings",
        "Test loading and inference in deployment environment",
        "Monitor memory usage in production"
    ]
}

print("💡 Model Optimization Best Practices:")
for category, practices in OPTIMIZATION_BEST_PRACTICES.items():
    print(f"\n{category.upper().replace('_', ' ')}:")
    for practice in practices:
        print(f"  ✓ {practice}")

## Key Takeaways

### What We Learned:
1. **Quantization dramatically reduces model size** (up to 75% reduction)
2. **Performance vs accuracy trade-offs** exist but are often acceptable
3. **Different quantization methods** suit different use cases
4. **Hardware considerations** are crucial for optimization choices

### Performance Summary:
- **4-bit quantization**: Best size reduction (~75% smaller)
- **8-bit quantization**: Good balance of size and quality
- **GPTQ**: Best performance for post-training quantization
- **Base FP32**: Highest quality, largest size

### Next Steps:
- Deploy optimized models using inference servers (vLLM, TGI)
- Implement monitoring and performance tracking
- Consider A/B testing different optimization levels
- Learn about model versioning and rollback strategies

### Resources:
- [BitsAndBytes Documentation](https://huggingface.co/docs/bitsandbytes/main/en/index)
- [GPTQ Paper](https://arxiv.org/abs/2210.17323)
- [HuggingFace Optimum](https://huggingface.co/docs/optimum/index)
- [vLLM Documentation](https://vllm.readthedocs.io/)